In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Base predictions: public blend by nina2025 (ps-s6e9-11, LB 0.94649)
# Post-processing idea adapted from public PS-S6E9 notebooks.
COMP_DIR  = Path('/kaggle/input/competitions/playground-series-s6e9')
BASE_FILE = Path('/kaggle/input/datasets/nina2025/ps-s6e9-11/0.94649.csv')
TARGET    = 'Will_Buy_EV'
GAMMA     = 0.74          # exponent applied to the 7th-decimal residual

def digit_residual_boost(p: np.ndarray, fine: int = 7, coarse: int = 6, gamma: float = GAMMA) -> np.ndarray:
    """Amplify the sub-precision residual between two rounding levels.
    Only reorders near-tied predictions -> affects AUC via tie-breaking."""
    hi = np.round(p, fine)
    r  = hi - np.round(p, coarse)
    return np.clip(hi + np.sign(r) * np.abs(r) ** gamma, 0.0, 1.0)

sub  = pd.read_csv(COMP_DIR / 'sample_submission.csv')
base = pd.read_csv(BASE_FILE)

assert len(base) == len(sub) and (base['id'].values == sub['id'].values).all(), "id mismatch"

sub[TARGET] = digit_residual_boost(base[TARGET].to_numpy(dtype=np.float64))
assert np.isfinite(sub[TARGET]).all()

sub.to_csv('submission.csv', index=False)

changed = (sub[TARGET].rank().values != base[TARGET].rank().values).mean()
print(f"Rows whose rank changed: {changed:.2%}")
print(sub.head(10))